# 🚀 Fine-Tuning BanglaBERT for Bangla Fake News Detection
**Model:** `csebuetnlp/banglabert` (ELECTRA Architecture pre-trained on 110GB Bangla text)

This notebook trains a state-of-the-art Transformer model on your Bangla fake news dataset using a **free GPU (T4)** on Google Colab.

### Steps to run:
1. In Google Colab, go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run all cells in order.
3. Upload your `bangla_train.csv` and `bangla_test.csv` when prompted.
4. At the end, download `bangla_bert_model.zip` and extract it into your project's `models/bangla_bert/` folder.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install required dependencies
!pip install -q transformers[torch] datasets accelerate scikit-learn seaborn matplotlib

In [ ]:
# 3. Upload bangla_train.csv and bangla_test.csv
from google.colab import files
import os

if not (os.path.exists('bangla_train.csv') and os.path.exists('bangla_test.csv')):
    print("Please upload 'bangla_train.csv' and 'bangla_test.csv':")
    uploaded = files.upload()
else:
    print("Datasets already found!")

In [ ]:
# 4. Load & Inspect Datasets
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

train_df = pd.read_csv('bangla_train.csv', encoding='utf-8-sig')
test_df = pd.read_csv('bangla_test.csv', encoding='utf-8-sig')

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

label2id = {'Real': 0, 'Fake': 1}
id2label = {0: 'Real', 1: 'Fake'}

train_texts = train_df['clean_text'].fillna('').astype(str).tolist()
train_labels = [label2id[str(lbl).strip()] for lbl in train_df['label']]

test_texts = test_df['clean_text'].fillna('').astype(str).tolist()
test_labels = [label2id[str(lbl).strip()] for lbl in test_df['label']]

# Create 15% validation split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.15, random_state=42, stratify=train_labels
)

print(f"Final splits -> Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

In [ ]:
# 5. Tokenize using BanglaBERT Tokenizer
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

MODEL_NAME = "csebuetnlp/banglabert"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LEN)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=MAX_LEN)

class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)
test_dataset = NewsDataset(test_encodings, test_labels)
print("Tokenization completed!")

In [ ]:
# 6. Setup Model, Metrics & Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'fake_precision': precision_score(labels, preds, pos_label=1, zero_division=0),
        'fake_recall': recall_score(labels, preds, pos_label=1, zero_division=0),
        'fake_f1': f1_score(labels, preds, pos_label=1, zero_division=0)
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='fake_f1',
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
# 7. Start Training (~5-8 minutes on T4 GPU)
trainer.train()

In [ ]:
# 8. Test Set Evaluation
test_results = trainer.predict(test_dataset)
logits = test_results.predictions
preds = np.argmax(logits, axis=1)

print("=" * 50)
print("TEST EVALUATION REPORT")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(test_labels, preds):.4f}")
print(f"Fake F1:   {f1_score(test_labels, preds, pos_label=1):.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, preds, target_names=['Real', 'Fake'], digits=4))

# Confusion Matrix Plot
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(test_labels, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('BanglaBERT Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# 9. Save and Download Trained Model Package
import shutil

save_dir = './bangla_bert_model'
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

shutil.make_archive('bangla_bert_model', 'zip', save_dir)
print("Model saved and zipped successfully!")

# Auto-download the zip file
files.download('bangla_bert_model.zip')